# Entrega Final: Procesamiento Acelerado de Lenguaje Natural
**Integrantes del grupo:**
- APONTE ZURITA WILLIAM ROMARIO
- VITERI AYALA FLAVIA KAMILA


Este notebook unifica las técnicas aprendidas a lo largo de las 3 semanas de clases para resolver el problema de clasificación de categorías de productos del dataset de MercadoLibre.

**El objetivo final es comparar el rendimiento (accuracy) de 4 métodos de NLP distintos:**
1. KNN con Embeddings Estáticos (FastText)
2. Zero-Shot Classification (RoBERTa-large-mnli)
3. Fine-Tuning de RoBERTa
4. KNN con Sentence-BERT (Embeddings Contextuales)


## Fase 1: Carga de Datos y Preprocesamiento


In [ ]:
import pandas as pd
import numpy as np
import string
import warnings
from time import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score
import gc

warnings.filterwarnings('ignore')
%matplotlib inline


In [ ]:
# Cargar datos
url = "https://raw.githubusercontent.com/fkviteri/uide-maestria/main/04-procesamiento-acelerado-lenguaje-natural/filter_data.csv"
data = pd.read_csv(url, index_col=0)
print(f"Dimensiones del dataset original: {data.shape}")


In [ ]:
# Normalización básica de NLP (Minúsculas y eliminación de puntuación)
# Nota: Los modelos Transformer manejan bien las stopwords, pero los métodos clásicos se benefician de la normalización.
def basic_normalization(sentence):
    punc = string.punctuation
    make_trans = str.maketrans('','',punc)
    sentence = sentence.lower()
    return sentence.translate(make_trans)

data['title_norm'] = data['title'].apply(basic_normalization)


### Creación de subconjuntos de prueba unificados
Para que la comparación entre los modelos sea justa y podamos usar los modelos Transformers (que son computacionalmente costosos), vamos a extraer un `test_set` de 100 muestras aleatorias. Todos los modelos serán evaluados EXCLUSIVAMENTE contra este `test_set`.


In [ ]:
np.random.seed(42)
# Generamos 100 índices aleatorios para test
test_indexs = np.random.randint(0, len(data), 100)

test_set = data.iloc[test_indexs].reset_index(drop=True)

# El resto será para entrenamiento
train_indexs = [i for i in range(len(data)) if i not in test_indexs]
train_set_full = data.iloc[train_indexs].reset_index(drop=True)

# Tomamos una muestra de train de 1000 registros para entrenar modelos sin colapsar la RAM
train_set = train_set_full.sample(1000, random_state=42).reset_index(drop=True)

print(f"Tamaño de entrenamiento: {train_set.shape[0]}")
print(f"Tamaño de prueba: {test_set.shape[0]}")


---
## Fase 2: Modelos Clásicos y Embeddings (FastText)
Vamos a entrenar un clasificador de KNN (K=5) utilizando los embeddings preentrenados de FastText para el idioma español.


In [ ]:
import urllib.request
import gzip
import shutil
import os
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter


In [ ]:
# Descargar y cargar modelo FastText (puede tardar un momento)
file_gz = "cc.es.300.vec.gz"
file_vec = "cc.es.300.vec"

if not os.path.exists(file_vec):
    print("Descargando modelo FastText...")
    url_fasttext = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.es.300.vec.gz"
    urllib.request.urlretrieve(url_fasttext, file_gz)
    print("Descomprimiendo...")
    with gzip.open(file_gz, "rb") as f_in:
        with open(file_vec, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
            
print("Cargando modelo...")
modelo_ft = KeyedVectors.load_word2vec_format(file_vec, binary=False)
print("Modelo FastText cargado.")


In [ ]:
def sentence_to_vector_ft(sentence, model):
    words = sentence.split()
    vectors = [model[word] for word in words if word in model]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Precomputar embeddings de entrenamiento FastText
print("Computando embeddings de entrenamiento...")
train_ft_embeddings = np.array([sentence_to_vector_ft(title, modelo_ft) for title in train_set['title_norm']])
train_categories = train_set['category'].tolist()

def knn_predict_ft(sentence, n=5):
    sentence_vector = sentence_to_vector_ft(sentence, modelo_ft).reshape(1, -1)
    # Cosine Similarity
    similitudes = cosine_similarity(sentence_vector, train_ft_embeddings)[0]
    top_n_idx = np.argsort(similitudes)[::-1][:n]
    votos = [train_categories[i] for i in top_n_idx]
    return max(set(votos), key=votos.count)


In [ ]:
# Evaluación FastText KNN
print("Evaluando modelo KNN (FastText)...")
preds_ft = []
for title in tqdm(test_set['title_norm'], desc="KNN FastText"):
    preds_ft.append(knn_predict_ft(title, n=5))

acc_fasttext = balanced_accuracy_score(test_set['category'], preds_ft)
print(f"\nAccuracy Balanced KNN FastText: {acc_fasttext*100:.2f}%")


In [ ]:
# Liberar memoria de FastText
del modelo_ft
del train_ft_embeddings
gc.collect()


---
## Fase 3: Modelos Transformers y Ajuste Fino (Fine-Tuning)
Ahora probaremos los modelos basados en la arquitectura Transformer, los cuales tienen comprensión contextual profunda.


### 3.1 RoBERTa: Zero-Shot Classification
Se utilizará el modelo sin ningún tipo de entrenamiento en nuestro dataset. Le pasaremos dinámicamente las categorías reales que existen en el `test_set`.


In [ ]:
from transformers import pipeline

print("Cargando pipeline Zero-Shot...")
classifier_zs = pipeline('zero-shot-classification', model='roberta-large-mnli')

# Obtenemos las categorías candidatas posibles dentro del test
categorias_candidatas = test_set['category'].unique().tolist()


In [ ]:
preds_zeroshot = []
for title in tqdm(test_set['title'], desc="Zero-Shot RoBERTa"):
    resultado = classifier_zs(title, candidate_labels=categorias_candidatas)
    preds_zeroshot.append(resultado['labels'][0])

acc_zeroshot = balanced_accuracy_score(test_set['category'], preds_zeroshot)
print(f"\nAccuracy Balanced Zero-Shot: {acc_zeroshot*100:.2f}%")


In [ ]:
# Liberamos la memoria del modelo zero-shot
del classifier_zs
gc.collect()


### 3.2 RoBERTa: Fine-Tuning
Ajustaremos el modelo `roberta-large-mnli` con nuestra pequeña muestra de entrenamiento (1000 registros).


In [ ]:
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

# Ajustar Encoder
label_encoder = LabelEncoder()
label_encoder.fit(data['category'])

# Preparar tensores
tokenizer = AutoTokenizer.from_pretrained("roberta-large-mnli")

train_encodings = tokenizer(train_set['title'].tolist(), truncation=True, padding=True, max_length=32)
test_encodings = tokenizer(test_set['title'].tolist(), truncation=True, padding=True, max_length=32)

y_train = label_encoder.transform(train_set['category'])
y_test = label_encoder.transform(test_set['category'])

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
test_dataset = TextDataset(test_encodings, y_test)


In [ ]:
os.environ["WANDB_DISABLED"] = "true"

num_labels = len(data['category'].unique())

# Importante: from_pretrained asegura Transfer Learning
model_ft = AutoModelForSequenceClassification.from_pretrained(
    "roberta-large-mnli", 
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)


In [ ]:
training_args = TrainingArguments(
    output_dir="./resultados_finales",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,  # 2 epochs para evitar sobrecalentamiento
    save_strategy="no",
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("Iniciando Fine-Tuning...")
trainer.train()


In [ ]:
# Evaluamos el fine-tuning
model_ft.eval()

preds_finetuned = []
for title in tqdm(test_set['title'], desc="Predicción Fine-Tuning"):
    inputs = tokenizer(title, truncation=True, padding=True, max_length=32, return_tensors="pt")
    inputs = {k: v.to(model_ft.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model_ft(**inputs).logits
    pred_idx = logits.argmax().item()
    preds_finetuned.append(label_encoder.inverse_transform([pred_idx])[0])

acc_finetuned = balanced_accuracy_score(test_set['category'], preds_finetuned)
print(f"\nAccuracy Balanced Fine-Tuned: {acc_finetuned*100:.2f}%")


In [ ]:
del model_ft
del trainer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()


### 3.3 KNN con Embeddings Contextuales (Sentence-BERT)
Utilizamos el modelo especializado `hiiamsid/sentence_similarity_spanish_es` para generar embeddings y buscar vecinos cercanos.


In [ ]:
from sentence_transformers import SentenceTransformer

# Cargamos el modelo BERT especializado en similitud en español
sbert_model = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')

# Precomputar embeddings BERT
print("Computando embeddings SBERT...")
train_sbert_embeddings = sbert_model.encode(train_set['title'].tolist())

def knn_predict_sbert(sentence, n=5):
    sentence_embedding = sbert_model.encode([sentence])
    similitudes = cosine_similarity(sentence_embedding, train_sbert_embeddings)[0]
    top_n_idx = np.argsort(similitudes)[::-1][:n]
    votos = [train_categories[i] for i in top_n_idx]
    return max(set(votos), key=votos.count)


In [ ]:
preds_sbert = []
for title in tqdm(test_set['title'], desc="KNN SBERT"):
    preds_sbert.append(knn_predict_sbert(title, n=5))

acc_sbert = balanced_accuracy_score(test_set['category'], preds_sbert)
print(f"\nAccuracy Balanced SBERT: {acc_sbert*100:.2f}%")


---
## Fase 4: Comparación Final de Rendimiento
Vamos a visualizar y comparar las métricas obtenidas por los 4 algoritmos.


In [ ]:
# Recopilamos las métricas
resultados = {
    'Modelo': [
        'KNN (FastText)', 
        'RoBERTa (Zero-Shot)', 
        'RoBERTa (Fine-Tuned)', 
        'KNN (Sentence-BERT)'
    ],
    'Accuracy (Balanced)': [
        acc_fasttext,
        acc_zeroshot,
        acc_finetuned,
        acc_sbert
    ]
}

df_resultados = pd.DataFrame(resultados)
df_resultados['Accuracy (Balanced)'] = df_resultados['Accuracy (Balanced)'] * 100
df_resultados.sort_values(by='Accuracy (Balanced)', ascending=False, inplace=True)
display(df_resultados)


In [ ]:
# Gráfico Comparativo
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='Accuracy (Balanced)', y='Modelo', data=df_resultados, palette='coolwarm')

# Añadir etiquetas a las barras
for i, v in enumerate(df_resultados['Accuracy (Balanced)']):
    ax.text(v + 0.5, i + 0.1, f"{v:.2f}%", color='black', fontweight='bold')

plt.title('Comparación de Rendimiento en Clasificación de Texto NLP', fontsize=15, pad=15)
plt.xlabel('Balanced Accuracy (%)', fontsize=12)
plt.ylabel('Arquitectura / Método', fontsize=12)
plt.xlim(0, max(df_resultados['Accuracy (Balanced)']) + 15)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


### Conclusiones de la Comparación
1. **Modelos KNN + Embeddings Contextuales (Sentence-BERT)**: Usualmente obtienen un excelente rendimiento porque están pre-entrenados para entender relaciones semánticas completas en el español, superando a los embeddings estáticos como FastText que pierden el contexto de la frase.
2. **RoBERTa Fine-Tuned**: Al especializarse directamente en las categorías de destino, demuestra un alto desempeño, justificando su coste computacional.
3. **Zero-Shot**: Sufre fuertemente porque el dominio (MercadoLibre) contiene categorías en inglés muy técnicas y los títulos están en español, evidenciando una brecha de dominio.
4. **FastText (Estático)**: Aunque captura similitudes sintácticas, la representación del vector global promediando la frase completa diluye las palabras clave.


---
## Fase 5: Aplicación en el Ámbito Laboral

**APONTE ZURITA WILLIAM ROMARIO**
En el sector de desarrollo de software Backend en Banco Pichincha, la clasificación automática de texto con modelos como RoBERTa podría aplicarse para la categorización y triaje automático de incidentes, bugs o tickets de soporte técnico. Al recibir reportes de errores de distintos canales, el modelo podría identificar a qué microservicio, equipo o dominio de negocio corresponde (ej. transferencias, pagos, tarjetas, cuentas) sin intervención manual, agilizando los tiempos de respuesta y resolución (MTTR).

El Fine-Tuning permitiría adaptar el modelo a la jerga técnica específica del banco, arquitecturas internas y códigos de error propios, entrenándolo con el histórico de incidencias de herramientas como Jira o ServiceNow. Esto aseguraría que el modelo entienda el contexto financiero y técnico simultáneamente.

Por otro lado, el clasificador de N-vecinos (KNN) basado en similitud semántica sería muy útil para identificar reportes de bugs duplicados o incidentes recurrentes. Al ingresar un nuevo ticket, el sistema podría buscar en la base de conocimientos los "n-vecinos" más cercanos semánticamente, sugiriendo a los desarrolladores backend soluciones de código previas, pull requests asociados o documentación técnica que resolvió problemas similares en el pasado.

**VITERI AYALA FLAVIA KAMILA**
En el contexto de la evaluación ex post de operaciones financiadas por CAF, la clasificación automática de texto con modelos como RoBERTa podría aplicarse para procesar grandes volúmenes de documentos de cierre de proyectos, informes de seguimiento y reportes de beneficiarios, clasificándolos automáticamente según dimensiones de evaluación como eficiencia, eficacia, sostenibilidad o impacto, sin necesidad de revisar manualmente cada documento. Esto reduciría significativamente el tiempo dedicado a la sistematización de información y permitiría evaluar un mayor número de operaciones en menos tiempo.

El Fine-Tuning permitiría especializar el modelo con la terminología propia de CAF y de la banca de desarrollo, entrenándolo con informes de evaluaciones previas ya clasificados. Así, el modelo aprendería a identificar automáticamente hallazgos relevantes, lecciones aprendidas o factores de riesgo en nuevos proyectos, facilitando la comparación sistemática entre operaciones de distintos sectores como infraestructura, educación o medio ambiente.

Finalmente, el clasificador de N-vecinos basado en similitud semántica podría usarse para identificar operaciones similares en el portafolio histórico de CAF, de modo que al iniciar una nueva evaluación ex post, el sistema recomiende automáticamente los marcos metodológicos, indicadores y criterios utilizados en proyectos comparables. Esto garantizaría mayor consistencia metodológica entre evaluaciones y facilitaría la identificación de patrones de éxito o fracaso a nivel de portafolio.
